## ***Lab 2: Unit Roots in Interest Rates***

In [1]:
!pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 25.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import pandas as pd
from scipy.stats import jarque_bera
from statsmodels.tsa.stattools import acf, pacf, q_stat
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron
from statsmodels.tsa.statespace.sarimax import SARIMAX
import plotly.graph_objects as go
from plotly.subplots import make_subplots

***Set default layout.***

In [3]:
default_layout = dict(
    xaxis=dict(titlefont=dict(size=25), tickfont=dict(size=20), showgrid=False),
    yaxis=dict(titlefont=dict(size=25), tickfont=dict(size=20), showgrid=False, zeroline=False),
    font=dict(family="Serif", size=20, color="black"),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(font=dict(size=18), bgcolor='rgba(255,255,255,0.5)', bordercolor='black', borderwidth=1),
    margin=dict(t=80, l=60, r=60, b=60)
)

# Functions

In [4]:
def SACF_SPACF_plotly(series, lag_max=24, ylim=(-0.15, 0.15)):
    # Compute ACF and PACF with confidence intervals
    acf_vals, acf_conf = acf(series, nlags=lag_max, alpha=0.05)
    pacf_vals, pacf_conf = pacf(series, nlags=lag_max, alpha=0.05, method='ols')

    lags = list(range(1, lag_max+1))

    fig = make_subplots(rows=1, cols=2, subplot_titles=("SACF", "SPACF"))

    # ACF: bars + CIs
    fig.add_trace(go.Bar(
        x=lags, y=acf_vals[1:], marker_color='blue', opacity=0.7,
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=lags, y=acf_conf[1:,0], mode='lines',
        line=dict(color='grey', width=2),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=lags, y=acf_conf[1:,1], mode='lines',
        line=dict(color='grey', width=2),
        fill='tonexty', fillcolor='rgba(128,128,128,0.15)',
        showlegend=False
    ), row=1, col=1)

    # PACF: bars + CIs
    fig.add_trace(go.Bar(
        x=lags, y=pacf_vals[1:], marker_color='blue', opacity=0.7,
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=lags, y=pacf_conf[1:,0], mode='lines',
        line=dict(color='grey', width=2),
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=lags, y=pacf_conf[1:,1], mode='lines',
        line=dict(color='grey', width=2),
        fill='tonexty', fillcolor='rgba(128,128,128,0.15)',
        showlegend=False
    ), row=1, col=2)

    # Layout
    fig.update_yaxes(range=ylim, row=1, col=1)
    fig.update_yaxes(range=ylim, row=1, col=2)

    fig.update_layout(
        title=dict(text='SACF and SPACF', x=0.5, font=dict(size=28, family='Serif', color='black')),
        font=dict(family="Serif", size=18, color="black"),
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(t=80, l=60, r=60, b=60),
        showlegend=False
    )

    fig.show()

In [5]:
def SACF_SPACF(series, lag_max = 24, alpha_level = 0.05, model_df = 0):
    """
    Compute the sample autocorrelation function (SACF), sample partial autocorrelation function (SPACF),
    and Ljung-Box Q-statistics for a time series.

    This function calculates the ACF and PACF values along with their corresponding confidence intervals
    for lags 1 through `lag_max` using the provided significance level (`alpha_level`). In addition, it
    computes the Ljung-Box Q-statistic and associated p-values (excluding lag 0). Set `model_df`
    to the number of dof lost.

    """

    # Calculate ACF and PACF with confidence intervals
    acf_vals, acf_confint = acf(series, nlags=lag_max, alpha=alpha_level)
    pacf_vals, pacf_confint = pacf(series, nlags=lag_max, alpha=alpha_level, method='ols')

    # Calculate Ljung-Box statistics and p-values
    lb_results = sm.stats.acorr_ljungbox(
        series,
        lags=range(1, lag_max + 1),
        model_df=model_df,
        return_df=True
    )

    # Build the results DataFrame
    df_acf_pacf = pd.DataFrame({
        "Lag": np.arange(1, lag_max + 1),
        "ACF": acf_vals[1:],
        "ACF_lower": acf_confint[1:, 0],
        "ACF_upper": acf_confint[1:, 1],
        "PACF": pacf_vals[1:],
        "PACF_lower": pacf_confint[1:, 0],
        "PACF_upper": pacf_confint[1:, 1],
        "Q-stat": lb_results["lb_stat"].values,
        "Q-stat Prob": lb_results["lb_pvalue"].values.round(6)
    })

    # Set the index to 'Lag' and extract the main columns
    df_acf_pacf.set_index("Lag", inplace=True)
    df_acf_pacf_small = df_acf_pacf[["ACF", "PACF", "Q-stat", "Q-stat Prob"]].copy()

    return df_acf_pacf_small

In [6]:
def extract_roots(results):
  ar_roots = results.arroots
  ar_roots_arr = np.array(ar_roots)
  modulus = np.abs(ar_roots_arr)
  return ar_roots_arr, modulus

In [7]:
def format_crit_vals(crit_dict):
  parts = []
  for k, v in crit_dict.items():
      formatted = f"{k}: {v:.4f}"
      parts.append(formatted)
  result = "; ".join(parts)
  return result

# 1) Data Exploration

In [8]:
df = pd.read_excel('/content/bund.xlsx', sheet_name='Monthly',header=0)
df

,observation_date,IRLTLT01DEM156N
0,1956-05-01,6.400000
1,1956-06-01,6.800000
2,1956-07-01,6.800000
3,1956-08-01,6.800000
4,1956-09-01,7.000000
...,...,...
832,2025-09-01,2.693182
833,2025-10-01,2.617826
834,2025-11-01,2.657500
835,2025-12-01,2.814211


In [9]:
df.rename(columns={
    'observation_date': 'date',
    'IRLTLT01DEM156N': 'annual_percent_yield_10Y'
}, inplace=True)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace = True)
df

,annual_percent_yield_10Y
date,
1956-05-01,6.400000
1956-06-01,6.800000
1956-07-01,6.800000
1956-08-01,6.800000
1956-09-01,7.000000
...,...
2025-09-01,2.693182
2025-10-01,2.617826
2025-11-01,2.657500


In [10]:
df.index.freq = 'MS' #set the frequency of the index to Month Start
df

,annual_percent_yield_10Y
date,
1956-05-01,6.400000
1956-06-01,6.800000
1956-07-01,6.800000
1956-08-01,6.800000
1956-09-01,7.000000
...,...
2025-09-01,2.693182
2025-10-01,2.617826
2025-11-01,2.657500


In [11]:
fig = make_subplots(
    rows=1, cols=1, shared_xaxes=True,
    subplot_titles=("10-Year Bund Yield",)
)

# Add trace for Bund yield
fig.add_trace(
    go.Scatter(
        x=df.index, y=df["annual_percent_yield_10Y"],
        mode='lines', line=dict(color='blue', width=2),
        name='Annualised 10-Year Yield'
    ),
    row=1, col=1
)

# Apply defaults
fig.update_layout(**default_layout, showlegend=False)

# Match subplot title style to your typography
fig.update_annotations(font=dict(size=30, family='Serif', color='black'))

# Y-axis formatting
fig.update_yaxes(title_text="Annualised Yield (%)",
                 titlefont=dict(size=25), tickfont=dict(size=20),
                 row=1, col=1)

# X-axis formatting
fig.update_xaxes(title_text="Date",
                 titlefont=dict(size=25), tickfont=dict(size=20),
                 tickformat="%Y-%m",
                 row=1, col=1)

fig.show()

Recall that, $$\hat{\rho}(h) = \frac{\displaystyle\sum_{t=h+1}^{T} (Y_t - \bar{Y})(Y_{t-h} - \bar{Y})}{\displaystyle\sum_{t=1}^{T} (Y_t - \bar{Y})^2}$$

where $\bar{Y} = \frac{1}{T}\sum_{t=1}^{T} Y_t$ is the sample mean and $h = 1, 2, \ldots$ is the lag.

In [12]:
yields = df['annual_percent_yield_10Y'].values
SACF_SPACF_df = SACF_SPACF(yields)
SACF_SPACF_df

,ACF,PACF,Q-stat,Q-stat Prob
Lag,,,,
1,0.997141,0.998132,835.207095,0.0
2,0.992697,-0.335446,1663.976994,0.0
3,0.987942,0.063200,2485.812079,0.0
4,0.982812,-0.094161,3300.110433,0.0
5,0.977230,-0.054028,4106.152636,0.0
6,0.971572,0.037348,4903.846604,0.0
7,0.965967,0.012996,5693.314025,0.0
8,0.960305,-0.007320,6474.494444,0.0
9,0.954531,-0.026245,7247.241859,0.0


In [13]:
SACF_SPACF_plotly(yields, ylim=(-1.1, 2.1))

# 2) Benchmark

In [14]:
df_CER = df.copy()
X = np.ones((len(df_CER['annual_percent_yield_10Y']), 1))
y = df_CER['annual_percent_yield_10Y']
model = sm.OLS(y, X)
results_sm = model.fit()
print(results_sm.summary())

                               OLS Regression Results                               
Dep. Variable:     annual_percent_yield_10Y   R-squared:                      -0.000
Model:                                  OLS   Adj. R-squared:                 -0.000
Method:                       Least Squares   F-statistic:                       nan
Date:                      Wed, 01 Jul 2026   Prob (F-statistic):                nan
Time:                              22:24:03   Log-Likelihood:                -2051.5
No. Observations:                       837   AIC:                             4105.
Df Residuals:                           836   BIC:                             4110.
Df Model:                                 0                                         
Covariance Type:                  nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [15]:
AIC = -2*-2051.5+2*1
BIC = -2*-2051.5+(np.log(837))*1
HQIC = (-2*-2051.5)+np.log((np.log(837)))*2*1
print(f'AIC:{AIC}')
print(f'BIC:{BIC}')
print(f'HQIC:{HQIC}')

AIC:4105.0
BIC:4109.7298240704895
HQIC:4106.813098004457


In [16]:
arma_0_0 = sm.tsa.statespace.SARIMAX(
    df,
    order=(0, 0, 0),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False,
)
results_arma_0_0 = arma_0_0.fit(method='powell',disp=False)
print(results_arma_0_0.summary())

                                  SARIMAX Results                                   
Dep. Variable:     annual_percent_yield_10Y   No. Observations:                  837
Model:                              SARIMAX   Log Likelihood               -2049.510
Date:                      Wed, 01 Jul 2026   AIC                           4103.020
Time:                              22:24:04   BIC                           4112.477
Sample:                          05-01-1956   HQIC                          4106.645
                               - 01-01-2026                                         
Covariance Type:                        opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      5.3619      0.108     49.705      0.000       5.150       5.573
sigma2         7.8873      0.522     15.097      0.000       6.863       8.911
Ljun

In [17]:
AIC = (-2*-2049.510)+(2*2)
BIC = (-2*-2049.510)+(np.log(837))*2
HQIC = (-2*-2049.510)+np.log((np.log(837)))*2*2
print(f'AIC:{AIC}')
print(f'BIC:{BIC}')
print(f'HQIC:{HQIC}')

AIC:4103.02
BIC:4112.479648140979
HQIC:4106.6461960089155


# 3) Autoregressive Model

An AR(2) model is defined as:

$$
Y_t = \phi_0 + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \varepsilon_t,
$$

where:

- $Y_t$ is the observed time series.
- $\varepsilon_t$ is white noise with zero mean and constant variance.
- $\phi_0, \phi_1, \phi_2$ are the parameters of the model.


In [18]:
arma_2_0 = SARIMAX(
    df,
    order=(2, 0, 0),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False)

results_arma_2_0 = arma_2_0.fit(method='powell',disp=False)

params_arma_2_0 = results_arma_2_0.params
phi0= params_arma_2_0['intercept']
phi1= params_arma_2_0['ar.L1']
phi2= params_arma_2_0['ar.L2']

print(results_arma_2_0.summary())

                                  SARIMAX Results                                   
Dep. Variable:     annual_percent_yield_10Y   No. Observations:                  837
Model:                     SARIMAX(2, 0, 0)   Log Likelihood                 255.414
Date:                      Wed, 01 Jul 2026   AIC                           -502.829
Time:                              22:24:04   BIC                           -483.919
Sample:                          05-01-1956   HQIC                          -495.579
                               - 01-01-2026                                         
Covariance Type:                        opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.0119      0.015      0.805      0.421      -0.017       0.041
ar.L1          1.3326      0.027     48.888      0.000       1.279       1.386
ar.L

In [19]:
ar_roots_arma_2_0, modulus_arma_2_0 = extract_roots(results_arma_2_0)
print(f'AR roots:{ar_roots_arma_2_0}')
print(f'Modulus of AR roots:{modulus_arma_2_0}')

AR roots:[1.00429657 2.96835108]
Modulus of AR roots:[1.00429657 2.96835108]


# 4) Forecasting

In [20]:
df_forecast = df.copy()

*Set up.*

In [21]:
train_size = int(len(df_forecast['annual_percent_yield_10Y']) * 0.95)
train_df = df_forecast['annual_percent_yield_10Y'].iloc[:train_size]
test_df  = df_forecast['annual_percent_yield_10Y'].iloc[train_size:]
n_forecasts = len(test_df)

In [22]:
actuals = test_df.values

# Out-of-Sample (OOS)

In [23]:
oos_start = test_df.index[0]
oos_end   = test_df.index[-1]

**Genuine OOS**

In [24]:
model_gen = SARIMAX(
    train_df,
    order=(2, 0, 0),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
)

In [25]:
res_gen = model_gen.fit(method='powell',disp=False)

*Parameter extraction.*

In [26]:
phi0_gen = res_gen.params['intercept']
phi1_gen = res_gen.params['ar.L1']
phi2_gen = res_gen.params['ar.L2']

*Forecast.*

In [27]:
gen_forecasts = np.zeros(n_forecasts)
gen_forecasts[0] = phi0_gen + phi1_gen * train_df.iloc[-1] + phi2_gen * train_df.iloc[-2]
gen_forecasts[1] = phi0_gen + phi1_gen * gen_forecasts[0] + phi2_gen * train_df.iloc[-1]

In [28]:
for t in range(2, n_forecasts):
    gen_forecasts[t] = phi0_gen + phi1_gen * gen_forecasts[t-1] + phi2_gen * gen_forecasts[t-2]

*Calculate forecast errors.*

In [29]:
actuals_gen = test_df.iloc[:n_forecasts].values
errors_gen  = actuals_gen - gen_forecasts

*Mean Absolute Forecast Error (MAFE)*

In [30]:
MAFE = np.mean(np.abs(errors_gen))

*Root Mean Square Forecast Error (RMSFE)*

In [31]:
RMSFE = np.sqrt(np.mean(errors_gen**2))

*Mean Absolute Percentage Error (MAPE)*

In [32]:
actuals_MAPE = np.where(actuals_gen == 0, np.nan, actuals_gen)

In [33]:
MAPE = np.mean(np.abs(errors_gen / actuals_MAPE)) * 100

*Percentage of correct sign predictions*

In [34]:
actual_changes   = np.diff(actuals_gen)
forecast_changes = np.diff(gen_forecasts)
correct_sign     = np.mean(np.sign(actual_changes) == np.sign(forecast_changes)) * 100

*Summary*

In [35]:
print(f"MAFE:                        {MAFE:.4f}")
print(f"RMSFE:                       {RMSFE:.4f}")
print(f"MAPE:                        {MAPE:.4f}%")
print(f"Correct sign predictions:    {correct_sign:.2f}%")

MAFE:                        1.3583
RMSFE:                       1.3862
MAPE:                        55.9598%
Correct sign predictions:    58.54%


**Recursive OOS**

*Forecast.*

In [36]:
rec_forecasts = np.zeros(n_forecasts)

In [37]:
for t in range(n_forecasts):
    expanding_window = pd.concat([
        train_df,
        test_df.iloc[:t]
    ])

    model_rec = SARIMAX(
        expanding_window,
        order=(2, 0, 0),
        trend='c',
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    res_rec = model_rec.fit(method='powell',disp=False)

    phi0_rec = res_rec.params['intercept']
    phi1_rec = res_rec.params['ar.L1']
    phi2_rec = res_rec.params['ar.L2']

    rec_forecasts[t] = (phi0_rec + phi1_rec * expanding_window.iloc[-1] + phi2_rec * expanding_window.iloc[-2])

*Calculate forecast errors.*

In [38]:
actuals_rec = test_df.iloc[:n_forecasts].values
errors_rec  = actuals_rec - rec_forecasts

*Mean Absolute Forecast Error (MAFE)*

In [39]:
MAFE = np.mean(np.abs(errors_rec))

*Root Mean Square Forecast Error (RMSFE)*

In [40]:
RMSFE = np.sqrt(np.mean(errors_rec**2))

*Mean Absolute Percentage Error (MAPE)*

In [41]:
actuals_MAPE = np.where(actuals_rec == 0, np.nan, actuals_rec)

In [42]:
MAPE = np.mean(np.abs(errors_gen / actuals_MAPE)) * 100

*Percentage of correct sign predictions*

In [43]:
actual_changes   = np.diff(actuals_rec)
forecast_changes = np.diff(rec_forecasts)
correct_sign     = np.mean(np.sign(actual_changes) == np.sign(forecast_changes)) * 100

*Summary*

In [44]:
print(f"MAFE:                        {MAFE:.4f}")
print(f"RMSFE:                       {RMSFE:.4f}")
print(f"MAPE:                        {MAPE:.4f}%")
print(f"Correct sign predictions:    {correct_sign:.2f}%")

MAFE:                        0.1376
RMSFE:                       0.1997
MAPE:                        55.9598%
Correct sign predictions:    56.10%


# In Sample (IS)

**IS**

In [45]:
model_is = SARIMAX(
    df_forecast['annual_percent_yield_10Y'],
    order=(2, 0, 0),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
)

In [46]:
res_is = model_is.fit(method='powell',disp=False)

*Parameter extraction.*

In [47]:
phi0_is = res_is.params['intercept']
phi1_is = res_is.params['ar.L1']
phi2_is = res_is.params['ar.L2']

*Unconditional mean.*

In [48]:
mu = phi0_is/ (1-phi1_is-phi2_is)

In [49]:
mu

np.float64(4.186978328194781)

*Forecast.*

In [50]:
is_forecasts = np.zeros(n_forecasts)
is_forecasts[0] = phi0_is + phi1_is * train_df.iloc[-1] + phi2_is * train_df.iloc[-2]
is_forecasts[1] = phi0_is + phi1_is * test_df.iloc[0] + phi2_is * train_df.iloc[-1]

In [51]:
for t in range(2, n_forecasts):
    is_forecasts[t] = phi0_is + phi1_is * test_df.iloc[t-1] + phi2_is * test_df.iloc[t-2]

*Calculate forecast errors.*

In [52]:
actuals_is = test_df.iloc[:n_forecasts].values
errors_is  = actuals_is - is_forecasts

*Mean Absolute Forecast Error (MAFE)*

In [53]:
MAFE = np.mean(np.abs(errors_is))

*Root Mean Square Forecast Error (RMSFE)*

In [54]:
RMSFE = np.sqrt(np.mean(errors_is**2))

*Mean Absolute Percentage Error (MAPE)*

In [55]:
actuals_MAPE = np.where(actuals_is == 0, np.nan, actuals_is)

In [56]:
MAPE = np.mean(np.abs(errors_is / actuals_MAPE)) * 100

*Percentage of correct sign predictions*

In [57]:
actual_changes   = np.diff(actuals_is)
forecast_changes = np.diff(is_forecasts)
correct_sign     = np.mean(np.sign(actual_changes) == np.sign(forecast_changes)) * 100

*Summary*

In [58]:
print(f"MAFE:                        {MAFE:.4f}")
print(f"RMSFE:                       {RMSFE:.4f}")
print(f"MAPE:                        {MAPE:.4f}%")
print(f"Correct sign predictions:    {correct_sign:.2f}%")

MAFE:                        0.1367
RMSFE:                       0.1986
MAPE:                        6.0556%
Correct sign predictions:    56.10%


# Forecasts Plots

**Plot the predictions.**

In [59]:
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Scatter(
    x=train_df.index, y=train_df.values,
    mode='lines',
    line=dict(color='grey', width=2),
    name='Train'
))

fig.add_trace(go.Scatter(
    x=test_df.index, y=test_df.values,
    mode='lines',
    line=dict(color='black', width=2),
    name='Actual (test)'
))

fig.add_trace(go.Scatter(
    x=test_df.index, y=gen_forecasts,
    mode='lines',
    line=dict(color='blue', width=2, dash='dash'),
    name='Genuine OOS'
))

fig.add_trace(go.Scatter(
    x=test_df.index, y=rec_forecasts,
    mode='lines',
    line=dict(color='red', width=2, dash='dash'),
    name='Recursive OOS'
))

fig.add_trace(go.Scatter(
    x=test_df.index, y=is_forecasts,
    mode='lines',
    line=dict(color='green', width=2, dash='dash'),
    name='In-Sample'
))

fig.add_trace(go.Scatter(
    x=df_forecast.index,
    y=np.full(len(df_forecast), mu),
    mode='lines',
    line=dict(color='orange', width=2, dash='dot'),
    name=f'Unconditional mean (μ={mu:.2f})'
))

fig.update_layout(
    **default_layout,
    title=dict(text='Forecast Comparison — Full Series', x=0.5),
)
fig.update_yaxes(title_text='Yield (%)', titlefont=dict(size=20), tickfont=dict(size=16))
fig.update_xaxes(title_text='Date',      titlefont=dict(size=20), tickfont=dict(size=16),
                 tickformat='%Y-%m')

fig.show()

#Summary Performances Forecasting Exercise

In [60]:
def compute_metrics(actuals, forecasts):
    errors = actuals - forecasts
    MAFE  = np.mean(np.abs(errors))
    RMSFE = np.sqrt(np.mean(errors**2))
    safe_actuals = np.where(actuals == 0, np.nan, actuals)
    MAPE  = np.nanmean(np.abs(errors / safe_actuals)) * 100
    actual_changes   = np.diff(actuals)
    forecast_changes = np.diff(forecasts)
    sign_pct = np.mean(np.sign(actual_changes) == np.sign(forecast_changes)) * 100
    return MAFE, RMSFE, MAPE, sign_pct

In [61]:
MAFE_gen, RMSFE_gen, MAPE_gen, sign_gen = compute_metrics(actuals_gen, gen_forecasts)
MAFE_rec, RMSFE_rec, MAPE_rec, sign_rec = compute_metrics(actuals_rec, rec_forecasts)
MAFE_is,  RMSFE_is,  MAPE_is,  sign_is  = compute_metrics(actuals_is,  is_forecasts)

In [62]:
summary = pd.DataFrame({
    'MAFE':            [MAFE_gen,  MAFE_rec,  MAFE_is],
    'RMSFE':           [RMSFE_gen, RMSFE_rec, RMSFE_is],
    'MAPE (%)':        [MAPE_gen,  MAPE_rec,  MAPE_is],
    'Correct Sign (%)': [sign_gen,  sign_rec,  sign_is],
}, index=['Genuine OOS', 'Recursive OOS', 'In-Sample'])

summary = summary.round(4)
print(summary.to_string())

                 MAFE   RMSFE  MAPE (%)  Correct Sign (%)
Genuine OOS    1.3583  1.3862   55.9598           58.5366
Recursive OOS  0.1376  0.1997    6.1073           56.0976
In-Sample      0.1367  0.1986    6.0556           56.0976


# 5) Unit Root Tests

# 10-Year Bund Yield

In [63]:
# ADF Test (BIC, constant only)
adf_bic = adfuller(df['annual_percent_yield_10Y'], autolag='BIC', regression='c')
print("ADF Test (BIC, constant only)")
print(f"ADF statistic: {adf_bic[0]:.4f}")
print(f"p-value:       {adf_bic[1]:.4f}")
print(f"Lags used:     {adf_bic[2]}")
print("Critical values:")
for k, v in adf_bic[4].items():
    print(f"    {k}: {v:.4f}")
print(f"IC Best (BIC): {adf_bic[5]:.4f}\n")

ADF Test (BIC, constant only)
ADF statistic: -1.2887
p-value:       0.6343
Lags used:     1
Critical values:
    1%: -3.4382
    5%: -2.8650
    10%: -2.5686
IC Best (BIC): -465.4104



In [64]:
# Phillips–Perron Test (constant only)
pp_test = PhillipsPerron(df['annual_percent_yield_10Y'], trend='c')
print("Phillips–Perron Test")
print(f"PP Statistic: {pp_test.stat:.4f}")
print(f"p-value:      {pp_test.pvalue:.4f}")
print("Critical values:")
# 'critical_values' is a dict
for k, v in pp_test.critical_values.items():
    print(f"    {k}: {v:.4f}")

Phillips–Perron Test
PP Statistic: -1.2796
p-value:      0.6385
Critical values:
    1%: -3.4382
    5%: -2.8650
    10%: -2.5686


In [65]:
# KPSS Test (constant only)
kpss_stat, kpss_pvalue, kpss_lags, kpss_critvals = kpss(df['annual_percent_yield_10Y'], regression='c')
print("KPSS Test")
print(f"Test Statistic: {kpss_stat:.4f}")
print(f"p-value:        {kpss_pvalue:.4f}")
print(f"Lags used:      {kpss_lags}")
print("Critical Values:", kpss_critvals)

KPSS Test
Test Statistic: 3.2531
p-value:        0.0100
Lags used:      18
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}


/tmp/ipykernel_13835/1275154651.py:2: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.




# Differenced 10-Year Bund Yield

We now difference the series.

In [66]:
differenced_df = df.copy()
differenced_df['differenced_annual_percent_yield_10Y'] = differenced_df['annual_percent_yield_10Y'].diff()
differenced_df.dropna(inplace=True)

In [67]:
fig = make_subplots(
    rows=1, cols=1, shared_xaxes=True,
    subplot_titles=("Differenced 10-Year Bund Yield",)
)

# Differenced yield trace
fig.add_trace(
    go.Scatter(
        x=differenced_df.index,
        y=differenced_df["differenced_annual_percent_yield_10Y"],
        mode='lines',
        line=dict(color='blue', width=2),
        name='Differenced Annualised 10-Year Yield'
    ),
    row=1, col=1
)

# Layout and styling
fig.update_layout(**default_layout, showlegend=True)
fig.update_annotations(font=dict(size=30, family='Serif', color='black'))

fig.update_yaxes(title_text="Differenced Annualised Yield (%)",
                 titlefont=dict(size=25), tickfont=dict(size=20),
                 row=1, col=1)
fig.update_xaxes(title_text="Date",
                 titlefont=dict(size=25), tickfont=dict(size=20),
                 tickformat="%Y-%m",
                 row=1, col=1)

fig.show()

In [68]:
differenced_yields = differenced_df['differenced_annual_percent_yield_10Y'].values
SACF_SPACF_df = SACF_SPACF(differenced_yields)
SACF_SPACF_df

,ACF,PACF,Q-stat,Q-stat Prob
Lag,,,,
1,0.333770,0.333771,93.467298,0.0
2,0.053993,-0.065386,95.916157,0.0
3,0.078477,0.091841,101.095624,0.0
4,0.097964,0.051350,109.176561,0.0
5,0.014362,-0.040028,109.350452,0.0
6,-0.022945,-0.015591,109.794826,0.0
7,-0.000959,0.004699,109.795604,0.0
8,0.027459,0.023558,110.433570,0.0
9,0.028214,0.019156,111.107917,0.0


In [69]:
SACF_SPACF_plotly(differenced_yields, ylim=[-0.2,0.5])

In [70]:
# ADF Test (BIC, constant only)
adf_bic = adfuller(differenced_yields, autolag='BIC', regression='c')
print("ADF Test (BIC, constant only)")
print(f"ADF statistic: {adf_bic[0]:.4f}")
print(f"p-value:       {adf_bic[1]:.4f}")
print(f"Lags used:     {adf_bic[2]}")
print("Critical values:")
for k, v in adf_bic[4].items():
    print(f"    {k}: {v:.4f}")
print(f"IC Best (BIC): {adf_bic[5]:.4f}\n")

ADF Test (BIC, constant only)
ADF statistic: -20.4610
p-value:       0.0000
Lags used:     0
Critical values:
    1%: -3.4382
    5%: -2.8650
    10%: -2.5686
IC Best (BIC): -469.0228



In [71]:
# Phillips–Perron Test (constant only)
pp_test = PhillipsPerron(differenced_yields, trend='c')
print("Phillips–Perron Test")
print(f"PP Statistic: {pp_test.stat:.4f}")
print(f"p-value:      {pp_test.pvalue:.4f}")
print("Critical values:")
# 'critical_values' is a dict
for k, v in pp_test.critical_values.items():
    print(f"    {k}: {v:.4f}")

Phillips–Perron Test
PP Statistic: -20.6208
p-value:      0.0000
Critical values:
    1%: -3.4382
    5%: -2.8650
    10%: -2.5686


In [72]:
# KPSS Test (constant only)
kpss_stat, kpss_pvalue, kpss_lags, kpss_critvals = kpss(differenced_yields, regression='c')
print("KPSS Test")
print(f"Test Statistic: {kpss_stat:.4f}")
print(f"p-value:        {kpss_pvalue:.4f}")
print(f"Lags used:      {kpss_lags}")
print("Critical Values:", kpss_critvals)

KPSS Test
Test Statistic: 0.0840
p-value:        0.1000
Lags used:      10
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}


/tmp/ipykernel_13835/2797061486.py:2: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.




# 6) Spurious Regression

In [73]:
temp = pd.read_csv('/content/bund1.csv', sep=r',', engine='python', header=0)
temp.rename(columns={
    'Date': 'date',
    'Yield': 'annual_percent_yield_1Y'
}, inplace=True)
temp = temp.dropna(axis=1, how='all')   # removes Unnamed columns
temp['date'] = pd.to_datetime(temp['date'])
temp.set_index('date', inplace = True)
yields = temp['annual_percent_yield_1Y'].values

/tmp/ipykernel_13835/1668950329.py:7: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [74]:
temp

,annual_percent_yield_1Y
date,
1994-02-01,5.430
1994-03-01,5.310
1994-04-01,5.130
1994-05-01,5.020
1994-06-01,5.130
...,...
2025-11-01,1.916
2025-12-01,2.025
2026-01-01,1.988


In [75]:
# ADF Test (BIC, constant only)
adf_bic = adfuller(yields, autolag='BIC', regression='c')
print("ADF Test (BIC, constant only)")
print(f"ADF statistic: {adf_bic[0]:.4f}")
print(f"p-value:       {adf_bic[1]:.4f}")
print(f"Lags used:     {adf_bic[2]}")
print("Critical values:")
for k, v in adf_bic[4].items():
    print(f"    {k}: {v:.4f}")
print(f"IC Best (BIC): {adf_bic[5]:.4f}\n")

ADF Test (BIC, constant only)
ADF statistic: -2.0364
p-value:       0.2708
Lags used:     3
Critical values:
    1%: -3.4483
    5%: -2.8694
    10%: -2.5710
IC Best (BIC): -185.1166



In [76]:
# Phillips–Perron Test (constant only)
pp_test = PhillipsPerron(yields, trend='c')
print("Phillips–Perron Test")
print(f"PP Statistic: {pp_test.stat:.4f}")
print(f"p-value:      {pp_test.pvalue:.4f}")
print("Critical values:")
# 'critical_values' is a dict
for k, v in pp_test.critical_values.items():
    print(f"    {k}: {v:.4f}")

Phillips–Perron Test
PP Statistic: -2.1119
p-value:      0.2398
Critical values:
    1%: -3.4481
    5%: -2.8694
    10%: -2.5709


In [77]:
# KPSS Test (constant only)
kpss_stat, kpss_pvalue, kpss_lags, kpss_critvals = kpss(yields, regression='c')
print("KPSS Test")
print(f"Test Statistic: {kpss_stat:.4f}")
print(f"p-value:        {kpss_pvalue:.4f}")
print(f"Lags used:      {kpss_lags}")
print("Critical Values:", kpss_critvals)

KPSS Test
Test Statistic: 1.7831
p-value:        0.0100
Lags used:      11
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}


/tmp/ipykernel_13835/2415607357.py:2: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.




In [78]:
df = df.join(temp, how='inner')

In [79]:
fig = make_subplots(
    rows=1, cols=1, shared_xaxes=True,
    subplot_titles=("1 and 10-Year Bund Yields",)
)

# 1-Year yield
fig.add_trace(
    go.Scatter(
        x=df.index, y=df["annual_percent_yield_1Y"],
        mode='lines',
        line=dict(color='orange', width=3),
        name="Annualised 1-Year Yield"
    ),
    row=1, col=1
)

# 10-Year yield
fig.add_trace(
    go.Scatter(
        x=df.index, y=df["annual_percent_yield_10Y"],
        mode='lines',
        line=dict(color='blue', width=2),
        name="Annualised 10-Year Yield"
    ),
    row=1, col=1
)

# Layout and styling
fig.update_layout(**default_layout, showlegend=True)
fig.update_annotations(font=dict(size=30, family='Serif', color='black'))

fig.update_yaxes(title_text="Annualised Yield (%)",
                 titlefont=dict(size=25), tickfont=dict(size=20),
                 row=1, col=1)
fig.update_xaxes(title_text="Date",
                 titlefont=dict(size=25), tickfont=dict(size=20),
                 tickformat="%Y-%m",
                 row=1, col=1)

fig.show()

In [80]:
X = df['annual_percent_yield_1Y']
y = df['annual_percent_yield_10Y']
X_with_constant = sm.add_constant(X)
model_spurious = sm.OLS(y, X_with_constant)
results_spurious = model_spurious.fit()
print(results_spurious.summary())

                               OLS Regression Results                               
Dep. Variable:     annual_percent_yield_10Y   R-squared:                       0.816
Model:                                  OLS   Adj. R-squared:                  0.815
Method:                       Least Squares   F-statistic:                     1626.
Date:                      Wed, 01 Jul 2026   Prob (F-statistic):          6.43e-137
Time:                              22:24:23   Log-Likelihood:                -493.62
No. Observations:                       369   AIC:                             991.2
Df Residuals:                           367   BIC:                             999.1
Df Model:                                 1                                         
Covariance Type:                  nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------

In [81]:
residuals_spurious = results_spurious.resid
residuals_acf_pacf = SACF_SPACF(residuals_spurious)
residuals_acf_pacf

,ACF,PACF,Q-stat,Q-stat Prob
Lag,,,,
1,0.981211,0.981397,358.160143,0.0
2,0.960145,-0.081686,702.040695,0.0
3,0.936646,-0.086899,1030.188654,0.0
4,0.910405,-0.084822,1341.057147,0.0
5,0.880987,-0.099623,1632.959890,0.0
6,0.848287,-0.089826,1904.341021,0.0
7,0.815703,0.011104,2155.967099,0.0
8,0.782759,-0.011912,2388.320600,0.0
9,0.747625,-0.068604,2600.872425,0.0


In [82]:
SACF_SPACF_plotly(residuals_spurious, ylim=[-1,1.2])

In [83]:
# ADF Test (BIC, constant only)
adf_bic = adfuller(residuals_spurious, autolag='BIC', regression='c')
print("ADF Test (BIC, constant only)")
print(f"ADF statistic: {adf_bic[0]:.4f}")
print(f"p-value:       {adf_bic[1]:.4f}")
print(f"Lags used:     {adf_bic[2]}")
print("Critical values:")
for k, v in adf_bic[4].items():
    print(f"    {k}: {v:.4f}")
print(f"IC Best (BIC): {adf_bic[5]:.4f}\n")

ADF Test (BIC, constant only)
ADF statistic: -1.8606
p-value:       0.3508
Lags used:     0
Critical values:
    1%: -3.4482
    5%: -2.8694
    10%: -2.5710
IC Best (BIC): -227.1938



In [84]:
# Phillips–Perron Test (constant only)
pp_test = PhillipsPerron(residuals_spurious, trend='c')
print("Phillips–Perron Test")
print(f"PP Statistic: {pp_test.stat:.4f}")
print(f"p-value:      {pp_test.pvalue:.4f}")
print("Critical values:")
# 'critical_values' is a dict
for k, v in pp_test.critical_values.items():
    print(f"    {k}: {v:.4f}")

Phillips–Perron Test
PP Statistic: -2.6307
p-value:      0.0868
Critical values:
    1%: -3.4482
    5%: -2.8694
    10%: -2.5710


In [85]:
# KPSS Test (constant only)
kpss_stat, kpss_pvalue, kpss_lags, kpss_critvals = kpss(residuals_spurious, regression='c')
print("KPSS Test")
print(f"Test Statistic: {kpss_stat:.4f}")
print(f"p-value:        {kpss_pvalue:.4f}")
print(f"Lags used:      {kpss_lags}")
print("Critical Values:", kpss_critvals)

KPSS Test
Test Statistic: 1.1626
p-value:        0.0100
Lags used:      11
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}


/tmp/ipykernel_13835/3606943491.py:2: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.




In [86]:
differenced_df_spurious = df.copy()
differenced_df_spurious['differenced_annual_percent_yield_1Y'] = differenced_df_spurious['annual_percent_yield_1Y'].diff()
differenced_df_spurious['differenced_annual_percent_yield_10Y'] = differenced_df_spurious['annual_percent_yield_10Y'].diff()
differenced_df_spurious.dropna(inplace=True)

In [87]:
X = differenced_df_spurious['differenced_annual_percent_yield_1Y']
y = differenced_df_spurious['differenced_annual_percent_yield_10Y']
X_with_constant = sm.add_constant(X)
differenced_model = sm.OLS(y, X_with_constant)
differenced_results = differenced_model.fit()
print(differenced_results.summary())

                                     OLS Regression Results                                     
Dep. Variable:     differenced_annual_percent_yield_10Y   R-squared:                       0.305
Model:                                              OLS   Adj. R-squared:                  0.303
Method:                                   Least Squares   F-statistic:                     160.5
Date:                                  Wed, 01 Jul 2026   Prob (F-statistic):           9.51e-31
Time:                                          22:24:24   Log-Likelihood:                 177.46
No. Observations:                                   368   AIC:                            -350.9
Df Residuals:                                       366   BIC:                            -343.1
Df Model:                                             1                                         
Covariance Type:                              nonrobust                                         
                              

In [88]:
differenced_residuals = differenced_results.resid
residuals_acf_pacf = SACF_SPACF(differenced_residuals)
residuals_acf_pacf

,ACF,PACF,Q-stat,Q-stat Prob
Lag,,,,
1,0.069637,0.069638,1.799151,0.179815
2,-0.031316,-0.035596,2.163981,0.338920
3,0.025272,0.029914,2.402239,0.493218
4,0.048698,0.044679,3.289345,0.510622
5,-0.000906,-0.005087,3.289653,0.655426
6,-0.053514,-0.049947,4.366802,0.627168
7,-0.001938,0.002948,4.368218,0.736522
8,0.031319,0.025715,4.739216,0.785051
9,0.016438,0.015866,4.841701,0.847883


In [89]:
SACF_SPACF_plotly(differenced_residuals, ylim=[-0.2,0.3])